In [1]:
import json

def read_jsonl(file_path):
    data_list = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data_list.append(json.loads(line.strip()))
    return data_list

def read_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

def write_json(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

def write_jsonl(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [2]:
def calculate_metrics(preds,labels,removal_index=[]):
    assert len(preds) == len(labels)
    metrics = {
        "recall":0,
        "perfect_recall":0,
    }
    cnt = 0

    for i, (pred, label) in enumerate(zip(preds, labels)):
        if i in removal_index:
            continue
        tgt = {(x.lower(), y.lower()) for x,y in label}
        prd = {(x.lower(), y.lower()) for x,y in pred}

        tp = len(prd & tgt)
        fp = len(prd - tgt)
        fn = len(tgt - prd)

        _recall =  tp / (tp + fn) if tp + fn != 0 else 0

        metrics['recall'] +=  _recall
        metrics['perfect_recall'] += tgt.issubset(prd)
        cnt+=1

    
    for key, value in metrics.items():
        if 'missing' not in key:
            metrics[key] = round(value/cnt*100,1)

    return metrics

In [3]:
prefix = "../data/label"

spider_answer = [p['table_id'] for p in read_jsonl(f"{prefix}/spider.jsonl")]
bird_answer = [p['table_id'] for p in read_jsonl(f"{prefix}/bird.jsonl")]
spider2_answer = [p['table_id'] for p in read_jsonl(f"{prefix}/spider2.jsonl")]

In [4]:
our_file_format = "{model}/{data}.jsonl"

contriever_spider = read_jsonl(our_file_format.format(model='contriever', data='spider'))
contriever_bird = read_jsonl(our_file_format.format(model='contriever', data='bird'))
contriever_spider2 = read_jsonl(our_file_format.format(model='contriever', data='spider2'))
uae_spider = read_jsonl(our_file_format.format(model='uae', data='spider'))
uae_bird = read_jsonl(our_file_format.format(model='uae', data='bird'))
uae_spider2 = read_jsonl(our_file_format.format(model='uae', data='spider2'))

In [7]:
def get_current_state():
    print(" ATR (Cont.) ")

    print("    [spider] ")
    performance = calculate_metrics(preds=contriever_spider, labels=spider_answer)
    for k, v in performance.items():
        print(f"        {k}: {v}%")
    print("    [bird] ")
    performance = calculate_metrics(preds=contriever_bird, labels=bird_answer)
    for k, v in performance.items():
        print(f"        {k}: {v}%")
    print("    [spider2] ")
    performance = calculate_metrics(preds=contriever_spider2, labels=spider2_answer)
    for k, v in performance.items():
        print(f"        {k}: {v}%")
    
    print()
    print(" ATR (UAE) ")    
    print("    [spider] ")
    performance = calculate_metrics(preds=uae_spider, labels=spider_answer)
    for k, v in performance.items():
        print(f"        {k}: {v}%")
    print("    [bird] ")
    performance = calculate_metrics(preds=uae_bird, labels=bird_answer)
    for k, v in performance.items():
        print(f"        {k}: {v}%")
    print("    [spider2] ")
    performance = calculate_metrics(preds=uae_spider2, labels=spider2_answer)
    for k, v in performance.items():
        print(f"        {k}: {v}%")

In [8]:
get_current_state()

 ATR (Cont.) 
    [spider] 
        recall: 99.5%
        perfect_recall: 99.2%
    [bird] 
        recall: 98.2%
        perfect_recall: 96.0%
    [spider2] 
        recall: 72.4%
        perfect_recall: 64.4%

 ATR (UAE) 
    [spider] 
        recall: 99.6%
        perfect_recall: 99.4%
    [bird] 
        recall: 98.6%
        perfect_recall: 97.1%
    [spider2] 
        recall: 75.4%
        perfect_recall: 68.7%
